In [46]:
import pandas as pd
import os

# --- 配置区 ---
dataset_list = ['20NG', 'thucnews', 'Yahoo', 'TREC', 'banking', 'stackoverflow','clinc', 'hwu', 'ecdt',  'mcid','DBPedia']

F_NUM = 5
P_NUM = 0
KNOWN_LABEL_RATIOS = [0.25, 0.5, 0.75]
LABELED_DATA_RATIOS = [0.1, 0.5, 1.0]

# --- 统计容器：显式区分三维 ---
rows = []
print("--- Step 2, Block 1: Calculating all statistics ---")

for dataset_name in dataset_list:
    try:
        # 整体样本量 / 标签总数（不区分 rate/LDR）
        total_train = len(pd.read_csv(f'{dataset_name}/origin_data/train.tsv', sep='\t'))
        total_dev   = len(pd.read_csv(f'{dataset_name}/origin_data/dev.tsv',   sep='\t'))
        total_test  = len(pd.read_csv(f'{dataset_name}/origin_data/test.tsv',  sep='\t'))
        total_labels= len(pd.read_csv(f'{dataset_name}/label/label.list', header=None))

        # rows.append(dict(dataset=dataset_name, known_label_ratio='ALL', labeled_data_ratio='ALL', split='train',  value=total_train))
        # rows.append(dict(dataset=dataset_name, known_label_ratio='ALL', labeled_data_ratio='ALL', split='dev',    value=total_dev))
        # rows.append(dict(dataset=dataset_name, known_label_ratio='ALL', labeled_data_ratio='ALL', split='test',   value=total_test))
        # rows.append(dict(dataset=dataset_name, known_label_ratio='ALL', labeled_data_ratio='ALL', split='\\#label', value=total_labels))

        # --- 在遍历 rate 之前，先为每个 LDR 写入 "满 known (ALL)" 的 train/dev/test/#label ---
        for LDR in LABELED_DATA_RATIOS:
            # \#label：总类数（不受 LDR 影响）
            rows.append(dict(
                dataset=dataset_name,
                known_label_ratio='ALL',
                labeled_data_ratio=LDR,
                split='\\#label',
                value=total_labels
            ))

            # train/dev：读取对应 LDR 的 labeled_data（若无 'labeled' 列则全计）
            for split in ['train', 'dev']:
                ldr_path = f'{dataset_name}/labeled_data/{LDR}/{split}.tsv'
                if os.path.exists(ldr_path):
                    df_ldr = pd.read_csv(ldr_path, sep='\t')
                    if 'labeled' in df_ldr.columns:
                        cnt = int((df_ldr['labeled'] == True).sum())
                    else:
                        cnt = len(df_ldr)
                    rows.append(dict(
                        dataset=dataset_name,
                        known_label_ratio='ALL',
                        labeled_data_ratio=LDR,
                        split=split,
                        value=cnt
                    ))
                # 若当前 LDR 缺该 split 文件则不写入（避免产生空列，后续本就会 drop 全空列）

            # test：若 LDR 下没有 test 文件，则回退到 origin_data 的 test 总数
            ldr_test_path = f'{dataset_name}/labeled_data/{LDR}/test.tsv'
            if os.path.exists(ldr_test_path):
                df_test_ldr = pd.read_csv(ldr_test_path, sep='\t')
                # test 一般无 labeled 列，直接全计
                test_cnt = len(df_test_ldr)
            else:
                test_cnt = total_test  # 回退
            rows.append(dict(
                dataset=dataset_name,
                known_label_ratio='ALL',
                labeled_data_ratio=LDR,
                split='test',
                value=test_cnt
            ))

        # 分 rate 统计
        for rate in KNOWN_LABEL_RATIOS:
            known_label_path = f'{dataset_name}/label/fold{F_NUM}/part{P_NUM}/label_known_{rate}.list'
            if not os.path.exists(known_label_path):
                print(f"    [警告] 缺少 {known_label_path}，跳过该 rate")
                continue
            known_labels = pd.read_csv(known_label_path, header=None)[0].tolist()
            known_cnt = len(known_labels)

            # 1) 在每个 LDR 下都写入一条 '#label'（即使没有 train/dev 文件也会显示）
            for LDR in LABELED_DATA_RATIOS:
                rows.append(dict(
                    dataset=dataset_name,
                    known_label_ratio=rate,
                    labeled_data_ratio=LDR,
                    split='\\#label',
                    value=known_cnt
                ))

                # 2) 分 LDR & split 的“已标注且属于已知类”的样本数
                for split in ['train', 'dev']:
                    labeled_data_path = f'{dataset_name}/labeled_data/{LDR}/{split}.tsv'
                    if not os.path.exists(labeled_data_path):
                        continue
                    df_labeled = pd.read_csv(labeled_data_path, sep='\t')
                    if 'labeled' in df_labeled.columns:
                        mask = df_labeled['label'].isin(known_labels) & (df_labeled['labeled'] == True)
                    else:
                        mask = df_labeled['label'].isin(known_labels)
                    rows.append(dict(
                        dataset=dataset_name,
                        known_label_ratio=rate,
                        labeled_data_ratio=LDR,
                        split=split,
                        value=int(mask.sum())
                    ))

    except FileNotFoundError as e:
        print(f"    [错误] 统计失败，缺少文件: {e.filename}，已跳过 {dataset_name}")
        continue

print("--- Data calculation complete. ---")
df = pd.DataFrame(rows)
df

--- Step 2, Block 1: Calculating all statistics ---
--- Data calculation complete. ---


,dataset,known_label_ratio,labeled_data_ratio,split,value
0,20NG,ALL,0.1,\#label,20
1,20NG,ALL,0.1,train,701
2,20NG,ALL,0.1,dev,96
3,20NG,ALL,0.1,test,2000
4,20NG,ALL,0.5,\#label,20
...,...,...,...,...,...
424,DBPedia,0.75,0.5,train,3111
425,DBPedia,0.75,0.5,dev,463
426,DBPedia,0.75,1.0,\#label,164
427,DBPedia,0.75,1.0,train,6195


In [47]:
# --- 构建 DataFrame 并做嵌套透视（先 LDR 再 rate） ---

pivot = df.pivot_table(
    index='dataset',
    columns=['labeled_data_ratio', 'known_label_ratio', 'split'],  # 顺序：LDR -> rate -> split
    values='value',
    aggfunc='sum'
)

# 1) 删除全为空值的列
pivot = pivot.dropna(axis=1, how='all')

# 2) 各层排序：数字在前，'ALL' 在后；split 顺序为 #label, train, dev, test
def _sort_key_num_all(v):
    if v == 'ALL':
        return (1, float('inf'))
    try:
        return (0, float(v))
    except:
        return (0, 0)

# 重新按排序后的多级列重建列索引（只包含目前存在的列）
lvl0_vals = sorted(pivot.columns.get_level_values(0).unique(), key=_sort_key_num_all)  # LDR
lvl1_vals = sorted(pivot.columns.get_level_values(1).unique(), key=_sort_key_num_all)  # rate
# split 层：仅保留现有的，并按优先序排列
split_current = list(pivot.columns.get_level_values(2).unique())
split_order = ['\\#label', 'train', 'dev', 'test']
lvl2_vals = [s for s in split_order if s in split_current] + [s for s in split_current if s not in split_order]

# 只对存在的组合进行笛卡尔积（避免引入空列）
from itertools import product
kept_cols = []
existing = set(pivot.columns)
for a, b, c in product(lvl0_vals, lvl1_vals, lvl2_vals):
    col = (a, b, c)
    if col in existing:
        kept_cols.append(col)

pivot = pivot.reindex(columns=pd.MultiIndex.from_tuples(kept_cols, names=pivot.columns.names))

# 3) 行也做一个自然排序
pivot = pivot.reindex(index=sorted(pivot.index))

# 4) 数字千分位格式化（保留 NaN）
pivot_fmt = pivot.applymap(lambda x: "{:,}".format(int(x)) if pd.notnull(x) else x)

pivot_fmt.to_excel('data_statics_nested.xlsx')
print("Saved to data_statics_nested.xlsx")
pivot_fmt

Saved to data_statics_nested.xlsx


/tmp/ipykernel_4139464/1747660759.py:45: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  pivot_fmt = pivot.applymap(lambda x: "{:,}".format(int(x)) if pd.notnull(x) else x)


labeled_data_ratio     0.1                                                   \
known_label_ratio     0.25               0.5               0.75               
split              \#label train dev \#label train  dev \#label  train  dev   
dataset                                                                       
20NG                     5   183  25      10   351   48      15    518   71   
DBPedia                 55   252  55     110   480  110     164    647  164   
TREC                    12   324  35      24   435   51      35    468   62   
Yahoo                    2   140  20       5   350   50       8    560   80   
banking                 19   229  28      38   472   55      58    704   81   
clinc                   38   457  76      75   901  150     112  1,345  224   
ecdt                     8    97  12      16   137   20      23    188   28   
hwu                     16   187  26      32   361   51      48    557   79   
mcid                     4    31   4       8    62    8      12     94   12   
stackoverflow            5   300  50      10   600  100      15    900  150   
thucnews                 4   200  28       7   350   49      10    500   70   

labeled_data_ratio          ...     1.0                                       \
known_label_ratio      ALL  ...     0.5                  0.75                  
split              \#label  ... \#label  train    dev \#label   train    dev   
dataset                     ...                                                
20NG                    20  ...      10  3,505    501      15   5,166    737   
DBPedia                219  ...     110  4,603    661     164   6,195    886   
TREC                    47  ...      24  4,350    475      35   4,683    513   
Yahoo                   10  ...       5  3,500    500       8   5,600    800   
banking                 77  ...      38  4,740    527      58   7,048    783   
clinc                  150  ...      75  9,000  1,125     112  13,440  1,680   
ecdt                    31  ...      16  1,372    130      23   1,879    180   
hwu                     64  ...      32  3,616    436      48   5,575    674   
mcid                    16  ...       8    601     85      12     905    127   
stackoverflow           20  ...      10  5,997    998      15   8,997  1,498   
thucnews                14  ...       7  3,500    502      10   5,000    716   

labeled_data_ratio                                
known_label_ratio      ALL                        
split              \#label   train    dev   test  
dataset                                           
20NG                    20   7,000  1,000  2,000  
DBPedia                219   7,000  1,000  2,000  
TREC                    47   4,849    532    490  
Yahoo                   10   7,000  1,000  2,000  
banking                 77   9,003  1,000  3,080  
clinc                  150  18,000  2,250  2,250  
ecdt                    31   2,099    200    770  
hwu                     64   7,712    933  1,032  
mcid                    16   1,198    170    331  
stackoverflow           20  11,996  1,998  5,991  
thucnews                14   7,000  1,000  1,421  

[11 rows x 39 columns]

In [48]:
# --- 透视：行= (dataset, labeled_data_ratio)，列= (known_label_ratio, split) ---
pivot = df.pivot_table(
    index=['labeled_data_ratio', 'dataset'],
    columns=['known_label_ratio', 'split'],
    values='value',
    aggfunc='sum'
)

# 1) 删除全为空值的列
pivot = pivot.dropna(axis=1, how='all')

# 2) 各层排序：数字在前，'ALL' 在后；split 顺序为 #label, train, dev, test
def _sort_key_num_all(v):
    if v == 'ALL':
        return (1, float('inf'))
    try:
        return (0, float(v))
    except Exception:
        return (0, 0)

# 2.1 列排序（列两层：known_label_ratio, split）
if isinstance(pivot.columns, pd.MultiIndex):
    # level 0: known_label_ratio
    lvl0_vals = sorted(pivot.columns.get_level_values(0).unique(), key=_sort_key_num_all)
    # level 1: split
    split_current = list(pivot.columns.get_level_values(1).unique())
    split_order = ['\\#label', 'train', 'dev', 'test']
    lvl1_vals = [s for s in split_order if s in split_current] + [s for s in split_current if s not in split_order]

    # 仅保留现有组合，按期望顺序重建
    from itertools import product
    kept_cols = []
    existing = set(pivot.columns)
    for a, b in product(lvl0_vals, lvl1_vals):
        col = (a, b)
        if col in existing:
            kept_cols.append(col)
    if kept_cols:
        pivot = pivot.reindex(columns=pd.MultiIndex.from_tuples(kept_cols, names=pivot.columns.names))

# 2.2 行排序（行两层：dataset, labeled_data_ratio；其中 labeled_data_ratio 按数字在前、'ALL' 在后）
def _sort_key_ldr(v):
    return _sort_key_num_all(v)

if isinstance(pivot.index, pd.MultiIndex):
    # 取出现有行索引，按 (dataset 字典序, labeled_data_ratio 的数字/ALL 规则) 排序
    idx_list = list(pivot.index)
    idx_list_sorted = sorted(idx_list, key=lambda t: (t[0], _sort_key_ldr(t[1])))
    pivot = pivot.reindex(index=idx_list_sorted)
else:
    # 单层索引兜底：自然排序
    pivot = pivot.sort_index()

# 3) 千分位格式化（保留 NaN）
pivot_fmt = pivot.applymap(lambda x: "{:,}".format(int(x)) if pd.notnull(x) else x)

# 4) 导出
pivot_fmt.to_excel('data_statics_nested.xlsx')
print("Saved to data_statics_nested.xlsx")

pivot_fmt

Saved to data_statics_nested.xlsx


/tmp/ipykernel_4139464/1797272048.py:55: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  pivot_fmt = pivot.applymap(lambda x: "{:,}".format(int(x)) if pd.notnull(x) else x)


known_label_ratio                   0.25                 0.5                \
split                            \#label  train  dev \#label  train    dev   
labeled_data_ratio dataset                                                   
0.1                20NG                5    183   25      10    351     48   
                   DBPedia            55    252   55     110    480    110   
                   TREC               12    324   35      24    435     51   
                   Yahoo               2    140   20       5    350     50   
                   banking            19    229   28      38    472     55   
                   clinc              38    457   76      75    901    150   
                   ecdt                8     97   12      16    137     20   
                   hwu                16    187   26      32    361     51   
                   mcid                4     31    4       8     62      8   
                   stackoverflow       5    300   50      10    600    100   
                   thucnews            4    200   28       7    350     49   
0.5                20NG                5    911  130      10  1,750    249   
                   DBPedia            55  1,204  177     110  2,315    341   
                   TREC               12  1,610  175      24  2,176    239   
                   Yahoo               2    700  100       5  1,750    250   
                   banking            19  1,150  130      38  2,368    265   
                   clinc              38  2,280  304      75  4,500    600   
                   ecdt                8    487   48      16    684     68   
                   hwu                16    943  110      32  1,808    214   
                   mcid                4    148   23       8    298     45   
                   stackoverflow       5  1,500  250      10  3,000    500   
                   thucnews            4  1,000  144       7  1,750    252   
1.0                20NG                5  1,825  261      10  3,505    501   
                   DBPedia            55  2,387  345     110  4,603    661   
                   TREC               12  3,224  350      24  4,350    475   
                   Yahoo               2  1,400  200       5  3,500    500   
                   banking            19  2,303  258      38  4,740    527   
                   clinc              38  4,560  570      75  9,000  1,125   
                   ecdt                8    976   92      16  1,372    130   
                   hwu                16  1,886  227      32  3,616    436   
                   mcid                4    298   43       8    601     85   
                   stackoverflow       5  2,999  499      10  5,997    998   
                   thucnews            4  2,000  287       7  3,500    502   

known_label_ratio                   0.75                    ALL          \
split                            \#label   train    dev \#label   train   
labeled_data_ratio dataset                                                
0.1                20NG               15     518     71      20     701   
                   DBPedia           164     647    164     219     736   
                   TREC               35     468     62      47     487   
                   Yahoo               8     560     80      10     700   
                   banking            58     704     81      77     900   
                   clinc             112   1,345    224     150   1,801   
                   ecdt               23     188     28      31     209   
                   hwu                48     557     79      64     770   
                   mcid               12      94     12      16     122   
                   stackoverflow      15     900    150      20   1,200   
                   thucnews           10     500     70      14     700   
0.5                20NG               15   2,581    366      20   3,498   
                   DBPedia           164   3,111    463     219   

In [49]:
data_statics = pivot_fmt.to_latex().replace(' ', '').replace('0000', '')
replace_map = {
    "Classes": '|Classes|',
    "train": '|Train|',
    "dev": '|Validation|',
    "test": '|Test|',
    "0000": "",
    ".00": "",
    # "lrrrrrrr": "l|p{1.2cm}p{1.2cm}p{1.2cm}p{1.2cm}p{1.2cm}p{1.2cm}p{1.2cm}",
    "toprule\n": "toprule\nDataset",
    "\nlabeled_data_ratio&dataset&&&&&&&&&&&&&\\\\": "",
    "known_label_ratio": "KCR",
    "labeled_data_ratio": "LAR",
    "split": "",
    " ": ""
}
for i, v in replace_map.items():
    data_statics = data_statics.replace(i, v)
with open('data_statics.latex', 'w') as w:
    w.write(data_statics)

In [50]:
import pandas as pd
import os
import re

def count_mixed_text(text: str) -> dict:
    """
    统计中英文混杂文本：
    - 英文：按单词数
    - 中文：按汉字数量
    """

    # 英文单词
    english_words = re.findall(r"[A-Za-z]+(?:'[A-Za-z]+)?", text)

    # 中文汉字（一个汉字算一个）
    chinese_chars = re.findall(r"[\u4e00-\u9fff]", text)

    return {
        "english_words": len(english_words),
        "chinese_words": len(chinese_chars),
        "total_words": len(english_words) + len(chinese_chars)
    }
    
    
# 读取 dataset -> language/domain 的映射表
df_type = pd.read_csv("data_type.csv")   # 确保文件和脚本在同一目录，或写绝对路径

rows = []

for _, row in df_type.iterrows():
    dataset_name = row['dataset']
    Dataset_name = row['Dataset']
    language = row['Language']
    domain = row['Text Domain']
    URL = row['URL']
    Scenario = row['Scenario']

    try:
        base_path = os.path.join(dataset_name, "origin_data")
        train_path = os.path.join(base_path, "train.tsv")
        dev_path   = os.path.join(base_path, "dev.tsv")
        test_path  = os.path.join(base_path, "test.tsv")

        # 合并 train/dev/test
        df_all = pd.concat([
            pd.read_csv(train_path, sep="\t"),
            pd.read_csv(dev_path, sep="\t"),
            pd.read_csv(test_path, sep="\t")
        ], ignore_index=True)

        # 标签数量
        num_labels = df_all['label'].nunique()

        # 平均字符长度
        avg_len = df_all['text'].astype(str).apply(lambda x: count_mixed_text(x)['total_words']).mean()
        avg_label_len = df_all['label'].astype(str).apply(lambda x: count_mixed_text(x)['total_words']).mean()

        rows.append({
            'Task': Scenario,
            'Text Domain': domain,
            'Dataset': Dataset_name,
            'Lang': language,
            '\#Instance': len(df_all),
            '\#Label': num_labels,
            'Avg. Instance': avg_len.round(2),
            'Avg. Label': avg_label_len.round(2),
            # 'Url': URL
        })

    except FileNotFoundError as e:
        print(f"[错误] 缺少文件 {e.filename}, 已跳过 {dataset_name}")
        continue

# 汇总表
df_stats = pd.DataFrame(rows).set_index('Task')
latex_code = df_stats.to_latex()
print(latex_code.replace('_', '\_').replace('0000', '').replace('& 1 &', '& 10000 &'))

<>:63: SyntaxWarning: invalid escape sequence '\#'
<>:64: SyntaxWarning: invalid escape sequence '\#'
<>:77: SyntaxWarning: invalid escape sequence '\_'
<>:63: SyntaxWarning: invalid escape sequence '\#'
<>:64: SyntaxWarning: invalid escape sequence '\#'
<>:77: SyntaxWarning: invalid escape sequence '\_'
/tmp/ipykernel_4139464/1149148028.py:63: SyntaxWarning: invalid escape sequence '\#'
  '\#Instance': len(df_all),
/tmp/ipykernel_4139464/1149148028.py:64: SyntaxWarning: invalid escape sequence '\#'
  '\#Label': num_labels,
/tmp/ipykernel_4139464/1149148028.py:77: SyntaxWarning: invalid escape sequence '\_'
  print(latex_code.replace('_', '\_').replace('0000', '').replace('& 1 &', '& 10000 &'))


\begin{tabular}{llllrrrr}
\toprule
 & Text Domain & Dataset & Lang & \#Instance & \#Label & Avg. Instance & Avg. Label \\
Task &  &  &  &  &  &  &  \\
\midrule
Topic Classification & Usenet News & 20NG & EN & 10000 & 20 & 290.41 & 2.80 \\
Topic Classification & Sina News & THUCNews & CN & 9421 & 14 & 749.07 & 2.00 \\
Topic Classification & Yahoo pages & Yahoo & EN & 10000 & 10 & 95.19 & 1.80 \\
Topic Classification & Twitter pages & X-Topic & Multilingual & 8587 & 49 & 26.86 & 1.00 \\
Intent Detection & Online banking queries & BANKING77 & EN & 13083 & 77 & 11.74 & 3.35 \\
Intent Detection & Conversational queries & CLINC150 & EN & 22500 & 150 & 8.23 & 1.94 \\
Intent Detection & Programming queries & StackOverflow & EN & 19985 & 20 & 8.36 & 1.05 \\
Intent Detection & Conversational queries & HWU64 & EN & 9677 & 64 & 6.60 & 2.12 \\
Intent Detection & Fact-based questiont & TREC & EN & 5871 & 47 & 8.72 & 2.00 \\
Intent Detection & Conversational queries & ECDT & CN & 3069 & 31 & 7.87 & 1